# 🧪 W2-D7 概念实验：第二周总复习（可执行版）

> 配套阅读：`第2周-Day7-第二周总复习.md`（知识全景图在那边）
> 第二周三条主线：**架构细节（LN/FFN）、推理优化（RoPE/KV Cache）、微调（QLoRA）**。
> 复习方式：每个主题做一个 5 分钟实验 + 最后 5 道可执行自测题
>
> 实验环境：纯 numpy + matplotlib。

## 实验 1：LayerNorm vs BatchNorm —— 归一化的"轴"不同

LayerNorm 按**样本**归一（每行 mean=0, std=1），与批大小、序列长度无关；
BatchNorm 按**特征维**归一（每列统计量依赖 batch）。
Transformer 序列长度可变、batch 内异质 —— 所以选 LN。

In [ ]:
import numpy as np

rng = np.random.default_rng(11)
X = rng.normal(loc=[1, 3, -2, 5, 0.5], scale=[0.5, 2, 1, 0.2, 3], size=(6, 5))  # 各列分布不同

def layer_norm(x, eps=1e-5):
    return (x - x.mean(axis=-1, keepdims=True)) / (x.std(axis=-1, keepdims=True) + eps)

def batch_norm(x, eps=1e-5):
    return (x - x.mean(axis=0, keepdims=True)) / (x.std(axis=0, keepdims=True) + eps)

ln, bn = layer_norm(X), batch_norm(X)
print("LayerNorm 每行(样本) 均值:", ln.mean(axis=1).round(6), " 标准差:", ln.std(axis=1).round(3))
print("BatchNorm 每列(特征) 均值:", bn.mean(axis=0).round(6))
print("\n→ LN: 每个样本独立归一，batch=1 也能用，序列长短无关 —— Transformer 的选择")
print("→ BN 的统计量跨样本共享：样本间异质（长短句混合）时会互相干扰")

## 实验 2：QLoRA 两件套 —— 4-bit 量化 + 低秩适配

**量化**：把 fp16 权重压到 4-bit（16 个级别），误差可控、存储省 4 倍。
**LoRA**：微调时不动 W，只学低秩增量 ΔW = A·B（rank r）。
用 SVD 验证"如果 ΔW 本身低秩，rank-8 的 A·B 就能精确表达"。

In [ ]:
# --- 量化部分 ---
W = rng.normal(size=(256, 256))

def quantize_uniform(w, bits=4):
    levels = 2**bits - 1
    lo, hi = w.min(), w.max()
    return np.round((w - lo) / (hi - lo) * levels) / levels * (hi - lo) + lo

Wq = quantize_uniform(W, bits=4)
rel_err_q = np.linalg.norm(W - Wq) / np.linalg.norm(W)
print(f"4-bit 量化相对误差: {rel_err_q:.2%}，存储 {16/4:.0f}× 压缩（QLoRA 的 NF4 更精细，误差更小）")

# --- LoRA 部分 ---
d_model, r = 256, 8
A = rng.normal(scale=0.02, size=(d_model, r))
B = rng.normal(scale=0.02, size=(r, d_model))
dW = A @ B                                   # 模拟微调产生的权重更新

S = np.linalg.svd(dW, compute_uv=False)
for rr in [1, 2, 4, 8, 16, 64]:
    err = np.sqrt((S[rr:]**2).sum() / (S**2).sum())
    print(f"rank={rr:>3} 重构误差: {err:8.2%}")

params_full = d_model * d_model
params_lora = 2 * d_model * r
print(f"\n全量微调 {params_full:,} 参数 vs LoRA(rank8) {params_lora:,} 参数 = {params_full/params_lora:.0f}× 节省")
print("→ QLoRA = 4-bit 冻结底座 + LoRA 只训 0.1% 参数：消费级显卡微调 7B 成为可能")

## 实验 3：低秩可视化 —— 微调增量 vs 随机矩阵的奇异值衰减

把"微调 ΔW"（低秩）和"随机矩阵"（满秩）的 rank-r 重构误差画在一起：
低秩矩阵误差骤降，随机矩阵几乎线性下降——LoRA 的前提是**微调增量天然低秩**。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

dW_random = rng.normal(size=(256, 256))
ranks = np.array([1, 2, 4, 8, 16, 32, 64, 128, 256])

def rank_errs(M):
    S = np.linalg.svd(M, compute_uv=False)
    return np.array([np.sqrt((S[rr:]**2).sum() / (S**2).sum()) for rr in ranks])

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(ranks, rank_errs(dW), "o-", label="微调增量 ΔW（rank-8 构造）")
ax.plot(ranks, rank_errs(dW_random), "s-", label="随机矩阵（满秩，无结构）")
ax.axvline(8, ls="--", color="gray", alpha=0.6)
ax.annotate("r=8: ΔW 误差≈0", xy=(8, 0.02), xytext=(30, 0.35),
            arrowprops=dict(arrowstyle="->"))
ax.set_xscale("log"); ax.set_xlabel("LoRA 秩 r"); ax.set_ylabel("rank-r 重构误差")
ax.set_title("LoRA 的前提：微调增量低秩，随机矩阵不是")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 实验 4：RoPE —— 为什么"旋转"能编码相对位置

把 q、k 的每两维当作平面上的向量，按位置 pos 旋转角度 pos·θ。
关键性质：`⟨R(q,m), R(k,n)⟩` 只依赖 **m−n**。
验证：固定相对距离、改变绝对位置，点积不变。

In [ ]:
def rope(x, pos):
    """对向量 x 的 (0,1),(2,3)... 维对按位置 pos 做平面旋转"""
    d = len(x)
    out = x.copy()
    for i in range(0, d, 2):
        theta = pos / (10000 ** (i / d))
        c, s = np.cos(theta), np.sin(theta)
        out[i], out[i+1] = c*x[i] - s*x[i+1], s*x[i] + c*x[i+1]
    return out

d = 16
q, k = rng.normal(size=d), rng.normal(size=d)

print("固定相对距离 m-n = 5，改变绝对位置：")
for m, n in [(5, 0), (105, 100), (1005, 1000), (5005, 5000)]:
    val = rope(q, m) @ rope(k, n)
    print(f"  m={m:>5}, n={n:>5}: q·k = {val:+.6f}")

vals = {rope(q, m) @ rope(k, m - 5) for m in [5, 105, 1005, 5005]}
print(f"\n四个点积是否全部相等: {len(vals) == 1}")
print("→ 相对位置 m-n 决定注意力分数，绝对位置无关：长文本外推更友好（对比绝对 PE）")

## 实验 5：可执行自测 —— 5 道题算出答案再对答案

In [ ]:
def check(no, desc, cond):
    print(f"  Q{no} {desc}: {'✓' if cond else '✗'}")

# Q1: LLaMA-7B（32层×32头×128维，fp16）在 2048 上下文的 KV Cache 有多大？
kv = 32 * 32 * 128 * 2048 * 2 * 2 / 1e9
check(1, f"≈{kv:.1f} GB（约 1GB 出头，与模型权重同量级）", 1.0 < kv < 1.2)

# Q2: 因果掩码矩阵是否下三角？
m = np.tril(np.ones((5, 5), dtype=int))
check(2, "因果掩码 = 下三角（含对角线）", bool((np.triu(m, 1) == 0).all()))

# Q3: fp16 → 4bit 存储压缩几倍？
check(3, "16/4 = 4 倍", 16 // 4 == 4)

# Q4: d=4096 的注意力投影用 LoRA rank=16 要训多少参数（相对全量）？
full, lora = 4096 * 4096, 2 * 4096 * 16
check(4, f"{lora/full:.2%}（约千分之八）", abs(lora / full - 0.0078) < 0.001)

# Q5: LayerNorm 沿哪个轴归一？
x = rng.normal(size=(3, 8))
lnx = (x - x.mean(axis=-1, keepdims=True)) / x.std(axis=-1, keepdims=True)
check(5, "每个样本一行独立归一（与 batch 无关）", bool(np.allclose(lnx.mean(axis=1), 0, atol=1e-8)))

print("\n第二周毕业 🎓 架构细节(LN/FFN) + 推理优化(RoPE/KVCache/GQA/Flash) + 微调(QLoRA) 全部串通")

## 结论

| 主题 | 实验 | 一句话结论 |
|---|---|---|
| LayerNorm | 1 | 按样本归一，与 batch/序列长度无关 |
| 4-bit 量化 | 2 | 相对误差 ~1%，存储省 4 倍 |
| LoRA | 2、3 | 微调增量低秩 → rank-8 重构误差≈0，参数省 128× |
| RoPE | 4 | 点积只依赖相对距离 → 外推友好 |
| 综合 | 5 | 5 道自测全 ✓ |

→ 深入阅读：同目录 `.md` 版本第三、四节（完整推理管线 + 知识图谱）